In [2]:
import yfinance as yf
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

pd.options.plotting.backend = "plotly"

In [3]:
tickers = ["RY.TO", "TD.TO", "BMO.TO", "BNS.TO", "CM.TO"]
bank_names = {
    "RY.TO": "Royal Bank",
    "TD.TO": "TD Bank",
    "BMO.TO": "Bank of Montreal",
    "BNS.TO": "Scotiabank",
    "CM.TO": "CIBC"
}

data = yf.download(tickers, start="2025-01-01", end="2026-08-04", group_by="ticker")
data.head()

[*********************100%***********************]  5 of 5 completed


Ticker         BNS.TO                                                 CM.TO  \
Price            Open       High        Low      Close    Volume       Open   
Date                                                                          
2025-01-02  71.035858  71.365619  70.550372  70.761055   9369600  86.211238   
2025-01-03  70.541215  71.503012  70.541215  70.651131  10726900  85.964914   
2025-01-06  71.035864  71.191583  70.422138  70.467941   7128800  85.974392   
2025-01-07  70.607257  70.672273  69.334808  69.390533   7238500  86.277559   
2025-01-08  69.214057  69.344095  68.693932  69.037590  10396800  85.604903   

Ticker                                                ...      BMO.TO  \
Price            High        Low      Close   Volume  ...        Open   
Date                                                  ...               
2025-01-02  86.514398  85.595442  85.661758  4485100  ...  131.433569   
2025-01-03  86.448078  85.358594  85.860703  2589600  ...  131.087456   
2025-01-06  86.362819  85.510179  85.803864  5476800  ...  130.563598   
2025-01-07  86.476507  84.932287  85.045967  4265700  ...  129.515861   
2025-01-08  86.324911  85.093319  86.031219  6830500  ...  130.591653   

Ticker                                                       TD.TO             \
Price             High         Low       Close   Volume       Open       High   
Date                                                                            
2025-01-02  132.013556  130.563588  130.769379  2036100  71.824745  71.834078   
2025-01-03  131.620667  129.600048  129.815216  1156500  71.638103  72.627250   
2025-01-06  130.881654  128.945224  129.048126  1750400  73.084492  73.261794   
2025-01-07  130.339079  129.179091  130.142624  1758100  72.617904  73.448409   
2025-01-08  133.285798  130.544877  132.415817  2985900  73.401765  73.485746   

Ticker                                      
Price             Low      Close    Volume  
Date                                        
2025-01-02  71.236854  71.386162  18507000  
2025-01-03  71.460809  72.580589  17948300  
2025-01-06  72.496609  72.580589  22054800  
2025-01-07  72.552583  73.233788  22302900  
2025-01-08  72.524600  72.711227  23749600  

[5 rows x 25 columns]

In [4]:
boc_dates = pd.to_datetime([
    "2025-01-29", "2025-03-12", "2025-04-16", "2025-06-04",
    "2025-07-30", "2025-09-17", "2025-10-29", "2025-12-10",
    "2026-01-28", "2026-03-18", "2026-04-29", "2026-06-10", "2026-07-15"
])

# Flag each date in the price data as an announcement day or not
all_dates = data.index
is_announcement = all_dates.isin(boc_dates)

announcement_flags = pd.Series(is_announcement, index=all_dates, name="is_announcement")
announcement_flags.sum()  # sanity check — should be close to 13, minus any that fell on a non-trading day

np.int64(13)

In [5]:
returns = pd.DataFrame(index=data.index)

for ticker in tickers:
    returns[ticker] = data[ticker]["Close"].pct_change() * 100  # % change day over day

returns["is_announcement"] = announcement_flags
returns.head()

,RY.TO,TD.TO,BMO.TO,BNS.TO,CM.TO,is_announcement
Date,,,,,,
2025-01-02,NaN,NaN,NaN,NaN,NaN,False
2025-01-03,0.952006,1.673192,-0.729653,-0.155346,0.232244,False
2025-01-06,0.195482,0.000000,-0.590909,-0.259287,-0.066199,False
2025-01-07,0.017245,0.899963,0.848131,-1.528933,-0.883289,False
2025-01-08,0.418856,-0.713551,1.746694,-0.508633,1.158494,False


In [6]:
# absolute move matters more than direction here — a -2% and +2% day are both "volatile"
abs_returns = returns[tickers].abs()
abs_returns["is_announcement"] = returns["is_announcement"]

comparison = abs_returns.groupby("is_announcement")[tickers].mean()
comparison.index = ["Normal Day", "Announcement Day"]
comparison

,RY.TO,TD.TO,BMO.TO,BNS.TO,CM.TO
Normal Day,0.763798,0.769009,0.873735,0.714330,0.856460
Announcement Day,0.765870,1.025612,0.860497,0.513124,0.715058


In [10]:
fig1 = go.Figure()

fig1.add_trace(go.Bar(
    x=[bank_names[t] for t in tickers],
    y=comparison.loc["Normal Day"],
    name="Normal Day",
    marker_color="#4C78A8"
))

fig1.add_trace(go.Bar(
    x=[bank_names[t] for t in tickers],
    y=comparison.loc["Announcement Day"],
    name="Announcement Day",
    marker_color="#F58518"
))

fig1.update_layout(
    title="Average Daily Price Move: Normal Days vs. BoC Announcement Days",
    yaxis_title="Average Absolute % Move",
    barmode="group",
    template="plotly_white",
    legend_title="Day Type"
)

fig1.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed